In [0]:
-- Create Quantexa Data Product organisations_all Schema metadata
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- metadata table
DROP TABLE IF EXISTS metadata;

CREATE OR REPLACE TABLE metadata ( 
	supplier STRING,
	data_product STRING,
	data_product_version STRING,
	schema_name STRING,
	schema_version STRING,
	schema_variant_name STRING,
	schema_variant_version STRING,
	instance STRING,
	instance_version STRING
 );

INSERT INTO metadata 
(
  supplier, 
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
) 
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.transactions_all',
  '0.1',
  'transactions_all.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);

In [0]:
-- Create Quantexa Data Product individual_customers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Aggregated Transactions table
DROP TABLE IF EXISTS hub_transaction;

CREATE OR REPLACE TABLE hub_transaction (
    transaction_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'Transaction Identity',
  	tennant_id BIGINT NOT NULL DEFAULT 0,
    bene_account_entity_id STRING,
    transaction_entity_id STRING,
    orig_account_entity_id STRING,
    bene_account_id BIGINT,
    orig_account_id BIGINT,
    transaction_id_raw STRING,
		period_start TIMESTAMP,
		period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE hub_transaction ADD CONSTRAINT pk_transaction_hubtransaction_transactionid PRIMARY KEY (transaction_id);


In [0]:
-- Create Quantexa Data Product individual_customers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Aggregated Transactions table
DROP TABLE IF EXISTS sat_transaction;

CREATE OR REPLACE TABLE sat_transaction (
    transaction_id BIGINT NOT NULL COMMENT 'Transactions Identity',
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    record_source_id BIGINT NOT NULL DEFAULT 0,
	  type_code STRING,
	  type_concept_id BIGINT,
	  type_raw_code STRING,
	  type_raw_concept_id BIGINT,
	  data_source_code STRING,
	  data_source_concept_id BIGINT,
	  data_source_raw_code STRING,
	  data_source_raw_concept_id BIGINT,
	  debit_or_credit_code STRING,
	  debit_or_credit_concept_id BIGINT,
	  debit_or_credit_raw_code STRING,
	  debit_or_credit_raw_concept_id BIGINT,
		counterparty_account_id STRING,
		counterparty_account_transaction_id STRING,
		is_self_transaction BOOLEAN,
		is_international_transaction BOOLEAN,
    amount DECIMAL(38,18),
		datetime TIMESTAMP,
    description STRING,
		balance_after DECIMAL(38,18),
		balance_before DECIMAL(38,18),	
	  payment_method_code STRING,
	  payment_method_concept_id BIGINT,
	  payment_method_raw_code STRING,
	  payment_method_raw_concept_id BIGINT,
    reference STRING,
		account_sort_code STRING,
		account_number STRING,
		account_number_suffix STRING,
    iban STRING,
		swiftbic STRING,
		beneficiary_name STRING,
	  currency_code STRING,
	  currency_concept_id BIGINT,
	  currency_raw_code STRING,
	  currency_raw_concept_id BIGINT,
	  country_code STRING,
	  country_concept_id BIGINT,
	  country_raw_code STRING,
	  country_raw_concept_id BIGINT,
    original_amount DECIMAL(38,18),
		original_account_number STRING,
		original_account_sort_code STRING,
		original_iban STRING,
		original_account_name STRING,
		original_currency_code STRING,
		original_currency_concept_id BIGINT,
	  original_currency_raw_code STRING,
	  original_currency_raw_concept_id BIGINT,
		original_country_code STRING,
		original_country_concept_id BIGINT,
	  original_country_raw_code STRING,
	  original_country_raw_concept_id BIGINT,
		original_bank STRING,
    original_bank_country_code STRING,
	  institution_bank STRING,
    institution_bank_country_code STRING,
	  sending_bank STRING,
    sending_bank_country_code STRING,
    beneficiary_bank STRING,
    beneficiary_bank_country_code STRING,
    payment_booking_date DATE,
	  payment_booking_system_code STRING,
	  payment_booking_system_concept_id BIGINT,
	  payment_booking_system_raw_code STRING,
	  payment_booking_system_raw_concept_id BIGINT,
	  payment_type_code STRING,
	  payment_type_concept_id BIGINT,
	  payment_type_raw_code STRING,
	  payment_type_raw_concept_id BIGINT,
		other_receiver_information STRING,
		payment_party_id BIGINT,
		payment_account_number STRING,
		payment_source_code STRING,
		period_start TIMESTAMP,
		period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_transaction ADD CONSTRAINT pk_transaction_sattransactions_transactionsid PRIMARY KEY (transaction_id);
ALTER TABLE sat_transaction ADD CONSTRAINT fk_transaction_sattransactions_hubtransaction_transactionid FOREIGN KEY (transaction_id) REFERENCES hub_transaction(transaction_id);


In [0]:
-- Create Quantexa Data Product individual_customers Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Transaction Party table
DROP TABLE IF EXISTS sat_transaction_party;

CREATE OR REPLACE TABLE sat_transaction_party (
    transaction_party_id BIGINT NOT NULL COMMENT 'Transaction Party Identity',
    transaction_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP(),
    record_source_id BIGINT NOT NULL DEFAULT 0,
	  type_code STRING,
	  type_concept_id BIGINT,
	  type_raw_code STRING,
	  type_raw_concept_id BIGINT,
	  rank INT,
		period_start TIMESTAMP,
		period_end TIMESTAMP
) USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_transaction_party ADD CONSTRAINT pk_transaction_sattransactionparty_transactionpartyid PRIMARY KEY (transaction_party_id);
ALTER TABLE sat_transaction_party ADD CONSTRAINT fk_transaction_sattransactionparty_hubtransactions_transactionid FOREIGN KEY (transaction_id) REFERENCES hub_transaction(transaction_id);


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_transactions AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  s.beneficiary_name,
	s.original_account_name  AS orignator_name,
 	s.amount,
	s.debit_or_credit_code,
	s.currency_code AS beneficary_currency_code,
	s.original_currency_code AS origator_currency_code,
  h.transaction_id,
  h.orig_account_entity_id,
  h.bene_account_entity_id,
  h.transaction_entity_id,
  h.bene_account_id,
  h.orig_account_id,
  h.transaction_id_raw,
  h.period_start AS hub_period_start,
  h.period_end AS hub_period_end,
	s.type_code,
	s.type_concept_id,
	s.type_raw_code,
	s.type_raw_concept_id,
	s.debit_or_credit_concept_id,
	s.debit_or_credit_raw_code,
	s.debit_or_credit_raw_concept_id,
	s.counterparty_account_id,
	s.counterparty_account_transaction_id,
	s.is_self_transaction,
	s.is_international_transaction,
	s.datetime,
	s.description,
	s.balance_after,
	s.balance_before,	
	s.payment_method_code,
	s.payment_method_concept_id,
	s.payment_method_raw_code,
	s.payment_method_raw_concept_id,
	s.reference,
	s.account_sort_code,
	s.account_number,
	s.account_number_suffix,
 	s.iban,
	s.swiftbic,
	s.currency_concept_id,
	s.currency_raw_code,
	s.currency_raw_concept_id,
	s.country_code,
	s.country_concept_id,
	s.country_raw_code,
	s.country_raw_concept_id,
	s.original_amount,
	s.original_account_number,
	s.original_account_sort_code,
	s.original_iban,
	s.original_currency_concept_id,
	s.original_currency_raw_code,
	s.original_currency_raw_concept_id,
	s.original_country_code,
	s.original_country_concept_id,
	s.original_country_raw_code,
	s.original_country_raw_concept_id,
	s.original_bank,
	s.original_bank_country_code,
	s.institution_bank,
	s.institution_bank_country_code,
	s.sending_bank,
	s.sending_bank_country_code,
	s.beneficiary_bank,
	s.beneficiary_bank_country_code,
	s.payment_booking_date,
	s.payment_booking_system_code,
	s.payment_booking_system_concept_id,
	s.payment_booking_system_raw_code,
	s.payment_booking_system_raw_concept_id,
	s.payment_type_code,
	s.payment_type_concept_id,
	s.payment_type_raw_code,
	s.payment_type_raw_concept_id,
	s.other_receiver_information,
	s.payment_party_id,
	s.payment_account_number,
	s.payment_source_code,	
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

  FROM demo_banking_silver.qdp_transactions_all.hub_transaction h
    JOIN demo_banking_silver.qdp_transactions_all.sat_transaction s
      ON h.transaction_id = s.transaction_id
    JOIN demo_banking_silver.qdp_refcodes_all.tennant t
      ON h.tennant_id = t.tennant_id
	ORDER BY s.beneficiary_name
;

SELECT * from view_transactions;
